# Sub-Agent RAG 구현 (Middleware 방식)

이 노트북은 메인 에이전트가 Deep Agents의 `SubAgentMiddleware`를 사용하여 RAG 전용 서브 에이전트에게 작업을 위임하는 구조를 구현합니다.

기존 코드(`04_sub_agent_rag.ipynb`)는 RAG 체인을 개발자가 직접 @tool로 래핑하여 메인 에이전트에게 수동으로 건네주는 방식이었습니다. 이 방식은 메인 에이전트의 컨텍스트 창(Context Window) 안에 서브 프로세스의 노이즈가 섞이거나, 복잡한 오케스트레이션 구조를 동적으로 유지하기 어렵다는 단점이 있었습니다.

##### 핵심 개선 사항
컨텍스트 격리 (Context Isolation): 메인 에이전트는 RAG 서브에이전트가 어떤 문서 검색 과정을 거쳤는지 세세하게 알 필요가 없습니다. 오직 **최종적으로 요약된 핵심 사실(Facts)**만 깔끔하게 수신하여 컨텍스트 윈도우 낭비와 환각(Hallucination)을 원천 차단합니다.
역할 정의의 정교화: RAG 서브에이전트에게 전용 Vector DB 검색 도구(search_company_guidebook)를 직접 쥐어주어, 서브에이전트가 지식 전문가(Subject Matter Expert)로서 독립적이고 효율적으로 지식을 탐색할 수 있도록 설계했습니다.
Deep Agents 표준 규격 준수: 공식 문서에서 제시하는 딕셔너리 기반의 선언적 서브에이전트 정의 스펙을 완벽하게 충족합니다.

In [16]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
import os

load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [17]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever()

In [18]:
from langchain.tools import tool

# 1. 서브 에이전트가 사용할 가이드북 검색 도구 정의
@tool
def search_company_guidebook(query: str) -> str:
    """테크노빌드 주식회사 사내 가이드북(복지, 자격증, 휴가 등)의 지식을 검색합니다."""
    docs = retriever.invoke(query)
    return "\n\n".join([doc.page_content for doc in docs])

In [19]:
# 2. RAG 서브 에이전트 스펙 설정
subagent_knowledge_expert = {
    "name": "knowledge_expert",
    "description": "테크노빌드 주식회사 사내 가이드북(복지, 자격증, 휴가 등)에 대한 전문적인 지식을 조회하고 핵심 사실을 추출합니다.",
    "system_prompt": """당신은 정보 추출 전문가입니다. 
제공된 검색 도구(search_company_guidebook)를 사용하여 사용자의 질문에 답하기 위해 필요한 **핵심 사실들만 목록 형식(-)**으로 추출하세요.
문장 형태의 완결된 답변이나 인삿말은 생략하고 정보만 전달하세요.""",
    "tools": [search_company_guidebook],
    "model": "google_genai:gemini-3-flash-preview",
    "middleware": [],
}

In [21]:
from langchain.agents import create_agent
# uv add deepagents
from deepagents.middleware.subagents import SubAgentMiddleware
# SubAgentMiddleware는 메인 에이전트가 서브에이전트(예: 우리가 만든 knowledge_expert)를 호출하여 복잡한 다단계 RAG나 추론을 수행하게 만듭니다.
# 이때 서브에이전트가 정보를 탐색하거나 결과 상태를 누적하고, 메인 에이전트와 통신하는 과정에서 "안전하게 읽고 쓸 수 있는 공용 가상 파일 공간(Workspace)"이 확보되어야 합니다.
# SubAgentMiddleware에 backend=StateBackend()를 건네주는 것은, 서브에이전트에게 "너희들이 이 세션 동안 자유롭고 안전하게 파일을 조작할 수 있는 샌드박스 상태 공간을 배정해 주겠다"라고 선언하는 것과 같습니다.
from deepagents.backends import StateBackend

# 3. 메인 에이전트 및 SubAgentMiddleware 정의
system_prompt = """당신은 테크노빌드의 통합 어시스턴트입니다.\n\
\n\
1. 정보 확인이 필요하면 서브에이전트인 'knowledge_expert'에게 작업을 위임하고(task 도구 사용), 그 결과를 확인하세요.\n\
2. 서브에이전트가 제공한 핵심 사실(Facts)들을 바탕으로, 사용자의 질문에 대해 친절하고 논리적인 완결된 문장으로 답변하세요.\n\
3. 문서에 없는 내용은 억지로 꾸며내지 마세요.\n\
"""

main_agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[],
    middleware=[
        SubAgentMiddleware(
            backend=StateBackend(),
            subagents=[subagent_knowledge_expert],
        )
    ],
    system_prompt=system_prompt
)

In [22]:
from langchain.messages import HumanMessage

# 4. 실행 테스트
query = "자격증 비용은 얼마를 받을 수 있어?"

res = main_agent.invoke({
    "messages": [HumanMessage(content=query)]
})

# 결과 추출 유틸리티 (03번 노트북 참고)
def extract_text(msg):
    if hasattr(msg, "content"):
        content = msg.content
        if isinstance(content, list) and len(content) > 0:
            if isinstance(content[0], dict) and "text" in content[0]:
                return content[0]["text"]
        return content
    return str(msg)

print("--- 최종 답변 ---")
print(extract_text(res["messages"][-1]))

--- 최종 답변 ---
테크노빌드에서는 직무와 관련된 국가 기술 자격증을 취득하실 경우, 등급에 따라 축하금(1회성)과 자격 수당(매월)을 지원하고 있습니다.

상세 금액은 다음과 같습니다:

*   **기술사/기능장:** 축하금 200만 원 / 월 수당 30만 원
*   **기사:** 축하금 50만 원 / 월 수당 10만 원
*   **산업기사:** 축하금 30만 원 / 월 수당 5만 원
*   **기능사:** 축하금 10만 원 / 월 수당 3만 원

**참고사항:**
*   축하금은 취득 횟수 제한 없이 지급되며, 증빙 서류 제출 후 2주 이내에 별도로 입금됩니다.
*   자격 수당은 매월 급여 명세서에 반영됩니다.
*   동일 등급 내 자격 수당은 1개만 인정되며, 상위 등급 자격증을 취득하실 경우 수당은 갱신됩니다.

추가로 궁금한 점이 있으시면 언제든 말씀해 주세요!


In [23]:
res

{'messages': [HumanMessage(content='자격증 비용은 얼마를 받을 수 있어?', additional_kwargs={}, response_metadata={}, id='e2ad7058-b56a-46f7-bee7-d7db209ffd4d'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'task', 'arguments': '{"description": "\\uc0ac\\ub0b4 \\uac00\\uc774\\ub4dc\\ubd81\\uc5d0\\uc11c \\uc790\\uaca9\\uc99d \\ucde8\\ub4dd \\ube44\\uc6a9 \\uc9c0\\uc6d0 \\uc815\\ucc45\\uc5d0 \\ub300\\ud574 \\uc870\\ud68c\\ud558\\uace0, \\uc9c0\\uc6d0 \\uac00\\ub2a5\\ud55c \\uae08\\uc561\\uc774\\ub098 \\uad00\\ub828 \\uaddc\\uc815\\uc744 \\uc694\\uc57d\\ud574\\uc11c \\uc54c\\ub824\\uc918.", "subagent_type": "knowledge_expert"}'}, '__gemini_function_call_thought_signatures__': {'4b0637a2-31d9-4d35-97d9-8a671c918968': 'EjQKMgEMOdbH+YW0v3IFTTL/YwxmDx4Kfk8or9aIzytT1OjB6WPoW+2t0twuymkLQ3aD4q0E'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019e38d9-897a-7d92-a08e-40ca92a8f73e-0

In [24]:
# 전체 메시지 흐름 확인 (도구 호출 과정 확인용)
for i, msg in enumerate(res["messages"]):
    print(f"[{i}] {type(msg).__name__}: {msg.content if not isinstance(msg.content, list) else 'Tool Call/Response'}")

[0] HumanMessage: 자격증 비용은 얼마를 받을 수 있어?
[1] AIMessage: Tool Call/Response
[2] ToolMessage: - 직무 관련 국가 기술 자격 취득 시 등급별 축하금(1회성) 및 자격 수당(매월) 지급
- **기술사/기능장:** 축하금 200만 원 / 수당 월 30만 원
- **기사:** 축하금 50만 원 / 수당 월 10만 원
- **산업기사:** 축하금 30만 원 / 수당 월 5만 원
- **기능사:** 축하금 10만 원 / 수당 월 3만 원
- 동일 등급 내 자격 수당은 1개만 인정(상위 등급 취득 시 갱신)
- 축하금은 취득 횟수 제한 없이 지급
- 자격 수당은 급여 명세서 반영, 축하금은 증빙 제출 후 2주 이내 별도 입금
[3] AIMessage: Tool Call/Response
